# Apriori Hiperlokal — Per Cabang + Per Segmen

**ML Engineer: Daffa (Product & Demand Strategy)**

Notebook ini menghasilkan aturan asosiasi produk spesifik per kombinasi **Cabang Gerai × Segmen Pelanggan × Status Member**.

## Pipeline

1. Load & join data: `df_basket_apriori` + `df_transaction_features` + inline K-Means segmen dari `df_rfm`
2. Nested loop: `for each branch × for each segment` → Apriori + Association Rules
3. Gabungkan semua aturan → satu DataFrame
4. Statistik ringkasan & export

## Input Files
- `df_basket_apriori.parquet` — matriks biner transaksi × 8 item menu
- `df_transaction_features.parquet` — info `transaction_id`, `city`, `user_id`, `member_status`
- `df_rfm.parquet` — RFM per `user_id` untuk inline K-Means segmentasi

## Output Files
- `df_rules_branch_segment.parquet` — semua aturan asosiasi
- `df_rules_branch_segment.csv` — versi CSV untuk review manual
- `apriori_branch_summary.json` — metadata ringkasan

## 1. Import Library

In [1]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import pyarrow.compute as pc
import gc
import json
import warnings
from pathlib import Path

from mlxtend.frequent_patterns import apriori, association_rules
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

print('Library imported.')

Library imported.


## 2. Load Data

In [2]:
gc.collect()

# ── 2a. Basket data (matriks biner) ───────────────────────────────────────
basket_table = pq.read_table('df_basket_apriori.parquet')
print(f'Basket table: {basket_table.num_rows:,} rows x {basket_table.num_columns} cols')

# ── 2b. Transaction features (cabang, member status, user_id) ─────────────
tx_table = pq.read_table(
    'df_transaction_features.parquet',
    columns=['transaction_id', 'city', 'user_id', 'member_status']
)
print(f'TX features: {tx_table.num_rows:,} rows')

# ── 2c. RFM data (untuk segmentasi inline) ────────────────────────────────
df_rfm = pd.read_parquet('df_rfm.parquet')
print(f'RFM data: {df_rfm.shape[0]:,} rows')

gc.collect()
print('\nData loaded successfully.')

Basket table: 9,064,669 rows x 9 cols


TX features: 14,623,691 rows
RFM data: 2,196,257 rows



Data loaded successfully.


## 3. Inline K-Means Segmentasi (dari RFM)

Karena file segmen (`df_member_with_segments.parquet`) belum tersedia,  
kita lakukan K-Means clustering langsung dari `df_rfm.parquet` menggunakan  
parameter yang konsisten dengan notebook K-Means utama.

In [3]:
gc.collect()

# ── 3a. Siapkan fitur RFM ──────────────────────────────────────────────────
rfm_features = df_rfm[['Recency', 'Frequency', 'Monetary']].copy()

# Log transform untuk mengatasi skewness
rfm_log = np.log1p(rfm_features)

# StandardScaler
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log)

print(f'RFM scaled shape: {rfm_scaled.shape}')

# ── 3b. MiniBatchKMeans (k=4, konsisten dengan K-Means Member notebook) ───
OPTIMAL_K = 4

kmeans = MiniBatchKMeans(
    n_clusters=OPTIMAL_K,
    batch_size=10000,
    random_state=42,
    n_init=10
)
cluster_labels = kmeans.fit_predict(rfm_scaled)

df_rfm['cluster'] = cluster_labels
print(f'Clustering done. Distribution:')
print(df_rfm['cluster'].value_counts().sort_index())

gc.collect()

RFM scaled shape: (2196257, 3)


Clustering done. Distribution:
cluster
0    701628
1    372646
2    645085
3    476898
Name: count, dtype: int64


22

In [4]:
# ── 3c. Beri label segmen berdasarkan profil RFM ──────────────────────────
cluster_profile = df_rfm.groupby('cluster').agg(
    R_mean=('Recency', 'mean'),
    F_mean=('Frequency', 'mean'),
    M_mean=('Monetary', 'mean'),
    count=('user_id', 'count')
).reset_index()

print('=== Cluster Profile ===')
print(cluster_profile.to_string(index=False))

# Auto-label berdasarkan posisi relatif RFM
r_med = cluster_profile['R_mean'].median()
f_med = cluster_profile['F_mean'].median()
m_med = cluster_profile['M_mean'].median()

cluster_name_map = {}
for _, row in cluster_profile.iterrows():
    c = int(row['cluster'])
    r, f, m = row['R_mean'], row['F_mean'], row['M_mean']
    
    if r < r_med and f > f_med and m > m_med:
        cluster_name_map[c] = 'Champions'
    elif r > r_med and f > f_med and m > m_med:
        cluster_name_map[c] = 'At Risk Regulars'
    elif r < r_med and f < f_med:
        cluster_name_map[c] = 'New Occasional'
    else:
        cluster_name_map[c] = 'Hibernating'

df_rfm['segment_name'] = df_rfm['cluster'].map(cluster_name_map)

print('\n=== Segment Map ===')
for k, v in cluster_name_map.items():
    cnt = (df_rfm['cluster'] == k).sum()
    print(f'  Cluster {k} → {v} ({cnt:,} members)')

# Buat lookup: user_id → segment_name
member_segment_lookup = df_rfm[['user_id', 'segment_name']].copy()
print(f'\nMember segment lookup: {len(member_segment_lookup):,} rows')

gc.collect()

=== Cluster Profile ===


 cluster     R_mean   F_mean     M_mean  count
       0 114.858794 2.986640  93.447937 701628
       1  12.873792 2.447403  75.597443 372646
       2 100.634005 1.140699  30.745983 645085
       3  83.453992 7.484810 236.596466 476898

=== Segment Map ===
  Cluster 0 → At Risk Regulars (701,628 members)
  Cluster 1 → New Occasional (372,646 members)
  Cluster 2 → Hibernating (645,085 members)
  Cluster 3 → Champions (476,898 members)

Member segment lookup: 2,196,257 rows


0

## 4. Join: Basket + Transaction Features + Segment

Menggabungkan data basket dengan informasi cabang (`city`), status member, dan segmen pelanggan.

In [5]:
gc.collect()

# ── 4a. Join basket + tx_features via PyArrow ──────────────────────────────
joined = basket_table.join(
    tx_table,
    keys='transaction_id',
    join_type='inner'
)
print(f'Basket+TX joined: {joined.num_rows:,} rows')

# Konversi ke pandas
df_joined = joined.replace_schema_metadata({}).to_pandas()

# Bersihkan memori PyArrow
del basket_table, tx_table, joined
gc.collect()

print(f'Columns: {list(df_joined.columns)}')
print(f'Cities: {sorted(df_joined["city"].unique())}')
print(f'Member status: {df_joined["member_status"].value_counts().to_dict()}')

Basket+TX joined: 9,064,669 rows


Columns: ['Americano', 'Cappuccino', 'Espresso', 'Flat White', 'Hot Chocolate', 'Latte', 'Matcha Latte', 'Mocha', 'transaction_id', 'city', 'user_id', 'member_status']


Cities: ['Alam Tun Hussein Onn', 'Bandar Seri Mulia', 'Damansara Saujana', 'Kampung Changkat', 'Kondominium Putra', 'PJS8', 'Seksyen 21', 'Taman Damansara', 'USJ 57W', 'USJ 89q']
Member status: {'Member': 4532595, 'Guest': 4532074}


In [6]:
# ── 4b. Attach segment label ──────────────────────────────────────────────
# Member: join via user_id → segment dari K-Means
# Guest: beri label 'Guest' (tidak ada RFM karena tidak ada user_id)

df_joined = df_joined.merge(
    member_segment_lookup,
    on='user_id',
    how='left'
)

# Guest/NaN user_id → segment = 'Guest'
df_joined['segment_name'] = df_joined['segment_name'].fillna('Guest')

print(f'Segment distribution:')
print(df_joined['segment_name'].value_counts())

del member_segment_lookup, df_rfm
gc.collect()

Segment distribution:
segment_name
Guest               4532074
Champions           2263621
At Risk Regulars    1321603
New Occasional       564980
Hibernating          382391
Name: count, dtype: int64


0

## 5. Apriori per Cabang × per Segmen

Nested loop mengiterasi semua kombinasi `branch_name × segment_name`.  
Threshold: minimal **100 transaksi** per sub-grup agar Apriori bermakna.

In [7]:
gc.collect()

# Konfigurasi Apriori
MIN_SUPPORT = 0.005
MIN_CONFIDENCE = 0.1
MIN_TRANSACTIONS = 100  # threshold minimum per subgroup

# Item columns (menu produk)
item_cols = ['Americano', 'Cappuccino', 'Espresso', 'Flat White',
             'Hot Chocolate', 'Latte', 'Matcha Latte', 'Mocha']

# Semua kombinasi
branches = sorted(df_joined['city'].unique())
segments = sorted(df_joined['segment_name'].unique())

print(f'Branches ({len(branches)}): {branches}')
print(f'Segments ({len(segments)}): {segments}')
print(f'Total kombinasi: {len(branches) * len(segments)}')
print(f'\nKonfigurasi: min_support={MIN_SUPPORT}, min_confidence={MIN_CONFIDENCE}, min_transactions={MIN_TRANSACTIONS}')

Branches (10): ['Alam Tun Hussein Onn', 'Bandar Seri Mulia', 'Damansara Saujana', 'Kampung Changkat', 'Kondominium Putra', 'PJS8', 'Seksyen 21', 'Taman Damansara', 'USJ 57W', 'USJ 89q']
Segments (5): ['At Risk Regulars', 'Champions', 'Guest', 'Hibernating', 'New Occasional']
Total kombinasi: 50

Konfigurasi: min_support=0.005, min_confidence=0.1, min_transactions=100


In [8]:
gc.collect()

all_rules = []
skipped_combos = []
combo_stats = []

for branch in branches:
    for segment in segments:
        # Filter sub-dataset
        mask = (df_joined['city'] == branch) & (df_joined['segment_name'] == segment)
        sub_df = df_joined.loc[mask, item_cols]
        n_tx = len(sub_df)
        
        # Cek minimum transaksi
        if n_tx < MIN_TRANSACTIONS:
            skipped_combos.append({
                'branch': branch,
                'segment': segment,
                'n_transactions': n_tx,
                'reason': f'Below threshold ({MIN_TRANSACTIONS})'
            })
            continue
        
        # Konversi ke bool
        sub_bool = sub_df.astype(bool)
        
        try:
            # Apriori: frequent itemsets
            freq_items = apriori(
                sub_bool,
                min_support=MIN_SUPPORT,
                use_colnames=True,
                low_memory=True
            )
            
            if len(freq_items) == 0:
                skipped_combos.append({
                    'branch': branch,
                    'segment': segment,
                    'n_transactions': n_tx,
                    'reason': 'No frequent itemsets found'
                })
                combo_stats.append({'branch': branch, 'segment': segment, 'n_tx': n_tx, 'n_rules': 0})
                continue
            
            # Association rules
            rules = association_rules(
                freq_items,
                metric='confidence',
                min_threshold=MIN_CONFIDENCE
            )
            
            if len(rules) > 0:
                rules['branch_name'] = branch
                rules['segment_name'] = segment
                
                # Konversi frozenset ke string
                rules['antecedents'] = rules['antecedents'].apply(lambda x: ', '.join(sorted(list(x))))
                rules['consequents'] = rules['consequents'].apply(lambda x: ', '.join(sorted(list(x))))
                
                all_rules.append(rules)
            
            combo_stats.append({'branch': branch, 'segment': segment, 'n_tx': n_tx, 'n_rules': len(rules)})
            
        except Exception as e:
            skipped_combos.append({
                'branch': branch,
                'segment': segment,
                'n_transactions': n_tx,
                'reason': f'Error: {str(e)}'
            })
            combo_stats.append({'branch': branch, 'segment': segment, 'n_tx': n_tx, 'n_rules': 0})
        
        gc.collect()

print(f'\nDone. Processed {len(combo_stats)} valid combinations.')
print(f'Skipped: {len(skipped_combos)} combinations.')


Done. Processed 50 valid combinations.
Skipped: 0 combinations.


## 6. Gabungkan & Ringkasan Statistik

In [9]:
# ── 6a. Gabungkan semua rules ─────────────────────────────────────────────
if len(all_rules) > 0:
    df_rules_all = pd.concat(all_rules, ignore_index=True)
    df_rules_all = df_rules_all.sort_values(['branch_name', 'segment_name', 'lift'], ascending=[True, True, False])
    df_rules_all = df_rules_all.reset_index(drop=True)
    print(f'Total aturan asosiasi: {len(df_rules_all):,}')
else:
    df_rules_all = pd.DataFrame()
    print('WARNING: Tidak ada aturan asosiasi yang dihasilkan!')

# ── 6b. Ringkasan per cabang ──────────────────────────────────────────────
df_stats = pd.DataFrame(combo_stats)

print('\n=== Aturan per Cabang ===')
branch_summary = df_stats.groupby('branch').agg(
    total_rules=('n_rules', 'sum'),
    total_transactions=('n_tx', 'sum'),
    combos_with_rules=('n_rules', lambda x: (x > 0).sum()),
    combos_without_rules=('n_rules', lambda x: (x == 0).sum())
).reset_index()
print(branch_summary.to_string(index=False))

# ── 6c. Cabang+Segmen dengan 0 aturan ────────────────────────────────────
zero_rules = df_stats[df_stats['n_rules'] == 0]
print(f'\n=== Kombinasi dengan 0 Aturan ({len(zero_rules)}) ===')
if len(zero_rules) > 0:
    print(zero_rules.to_string(index=False))
else:
    print('Semua kombinasi menghasilkan aturan!')

# ── 6d. Top 5 rules per cabang ────────────────────────────────────────────
if len(df_rules_all) > 0:
    print('\n=== Top 5 Rules per Cabang (by Confidence) ===')
    for branch in branches:
        branch_rules = df_rules_all[df_rules_all['branch_name'] == branch]
        if len(branch_rules) == 0:
            print(f'\n{branch}: Tidak ada aturan')
            continue
        top5 = branch_rules.nlargest(5, 'confidence')
        print(f'\n{branch} ({len(branch_rules)} rules):')
        display_cols = ['antecedents', 'consequents', 'segment_name', 'support', 'confidence', 'lift']
        available_cols = [c for c in display_cols if c in top5.columns]
        print(top5[available_cols].to_string(index=False))

Total aturan asosiasi: 9,048

=== Aturan per Cabang ===
              branch  total_rules  total_transactions  combos_with_rules  combos_without_rules
Alam Tun Hussein Onn          888              907480                  5                     0
   Bandar Seri Mulia          908              906742                  5                     0
   Damansara Saujana          912              905789                  5                     0
    Kampung Changkat          915              906485                  5                     0
   Kondominium Putra          916              906163                  5                     0
                PJS8          906              905261                  5                     0
          Seksyen 21          877              907405                  5                     0
     Taman Damansara          914              907320                  5                     0
             USJ 57W          908              906558                  5                 

In [10]:
# ── 6e. Skipped combinations detail ──────────────────────────────────────
if len(skipped_combos) > 0:
    print('=== Skipped Combinations ===')
    df_skipped = pd.DataFrame(skipped_combos)
    print(df_skipped.to_string(index=False))

## 7. Export Results

In [11]:
gc.collect()

if len(df_rules_all) > 0:
    # ── 7a. Parquet ───────────────────────────────────────────────────────
    df_rules_all.to_parquet('df_rules_branch_segment.parquet', index=False)
    print('Saved: df_rules_branch_segment.parquet')
    
    # ── 7b. CSV ───────────────────────────────────────────────────────────
    df_rules_all.to_csv('df_rules_branch_segment.csv', index=False)
    print('Saved: df_rules_branch_segment.csv')
    
    # ── 7c. Summary JSON ──────────────────────────────────────────────────
    summary = {
        'total_rules': int(len(df_rules_all)),
        'total_branches': int(len(branches)),
        'total_segments': int(len(segments)),
        'min_support': MIN_SUPPORT,
        'min_confidence': MIN_CONFIDENCE,
        'min_transactions_threshold': MIN_TRANSACTIONS,
        'rules_per_branch': branch_summary.set_index('branch')['total_rules'].to_dict(),
        'skipped_combinations': len(skipped_combos),
        'segment_names': segments,
        'branch_names': branches,
        'cluster_to_segment_map': {str(k): v for k, v in cluster_name_map.items()}
    }
    
    with open('apriori_branch_summary.json', 'w', encoding='utf-8') as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)
    print('Saved: apriori_branch_summary.json')
    
    print(f'\nTotal output: {len(df_rules_all):,} rules across {len(branches)} branches and {len(segments)} segments.')
else:
    print('WARNING: Tidak ada rules untuk disimpan!')

Saved: df_rules_branch_segment.parquet
Saved: df_rules_branch_segment.csv
Saved: apriori_branch_summary.json

Total output: 9,048 rules across 10 branches and 5 segments.


## 8. Assertions & Validasi

In [12]:
# ── Automated Assertions ──────────────────────────────────────────────────
print('=== Running Assertions ===')

# 1. Total baris > 0
assert len(df_rules_all) > 0, 'FAIL: Tidak ada aturan yang dihasilkan'
print(f'[PASS] Total rules: {len(df_rules_all):,}')

# 2. Kolom branch_name memiliki cabang-cabang yang benar
n_branches_with_rules = df_rules_all['branch_name'].nunique()
print(f'[INFO] Branches with rules: {n_branches_with_rules}/{len(branches)}')

# 3. Tidak ada NaN di kolom metrik
for col in ['support', 'confidence', 'lift']:
    nan_count = df_rules_all[col].isna().sum()
    assert nan_count == 0, f'FAIL: {col} has {nan_count} NaN values'
    print(f'[PASS] {col}: no NaN')

# 4. Support antara 0 dan 1
assert df_rules_all['support'].between(0, 1).all(), 'FAIL: support out of range [0,1]'
print('[PASS] support in range [0,1]')

# 5. Confidence antara 0 dan 1
assert df_rules_all['confidence'].between(0, 1).all(), 'FAIL: confidence out of range [0,1]'
print('[PASS] confidence in range [0,1]')

# 6. Lift > 0
assert (df_rules_all['lift'] > 0).all(), 'FAIL: lift has non-positive values'
print('[PASS] lift > 0')

# 7. File output tersedia
for f in ['df_rules_branch_segment.parquet', 'df_rules_branch_segment.csv', 'apriori_branch_summary.json']:
    assert Path(f).exists(), f'FAIL: {f} not found'
    print(f'[PASS] {f} exists')

print('\n=== All Assertions Passed ===')

=== Running Assertions ===
[PASS] Total rules: 9,048
[INFO] Branches with rules: 10/10
[PASS] support: no NaN
[PASS] confidence: no NaN
[PASS] lift: no NaN
[PASS] support in range [0,1]
[PASS] confidence in range [0,1]
[PASS] lift > 0
[PASS] df_rules_branch_segment.parquet exists
[PASS] df_rules_branch_segment.csv exists
[PASS] apriori_branch_summary.json exists

=== All Assertions Passed ===


In [13]:
# Final summary
print('=' * 60)
print('APRIORI HIPERLOKAL — SELESAI')
print('=' * 60)
print(f'Total aturan: {len(df_rules_all):,}')
print(f'Cabang: {n_branches_with_rules}')
print(f'Segmen: {df_rules_all["segment_name"].nunique()}')
print(f'Avg support: {df_rules_all["support"].mean():.4f}')
print(f'Avg confidence: {df_rules_all["confidence"].mean():.3f}')
print(f'Avg lift: {df_rules_all["lift"].mean():.3f}')
print(f'\nFiles exported:')
print(f'  - df_rules_branch_segment.parquet')
print(f'  - df_rules_branch_segment.csv')
print(f'  - apriori_branch_summary.json')

gc.collect()
print('\nDone.')

APRIORI HIPERLOKAL — SELESAI
Total aturan: 9,048
Cabang: 10
Segmen: 5
Avg support: 0.0231
Avg confidence: 0.136
Avg lift: 0.463

Files exported:
  - df_rules_branch_segment.parquet
  - df_rules_branch_segment.csv
  - apriori_branch_summary.json

Done.
